# Lakebase Search execution evidence

Calls the deployed Nimbus app with a short-lived token obtained from a named Databricks CLI profile. The token is held only in memory and is never printed or stored in notebook output.

In [1]:
import json, os, subprocess, urllib.parse, urllib.request

PROFILE = os.environ.get("DATABRICKS_CONFIG_PROFILE", "fe-sandbox-last-penguin")
APP_NAME = "nimbus-growth-desk"

def cli_json(*args):
    raw = subprocess.check_output(["databricks", *args, "--profile", PROFILE, "-o", "json"], text=True)
    return json.loads(raw)

app = cli_json("apps", "get", APP_NAME)
token = cli_json("auth", "token")["access_token"]
url = app["url"].rstrip("/") + "/api/search-experiments?" + urllib.parse.urlencode({"q": "checkout android gen-z", "limit": 5})
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
with urllib.request.urlopen(request, timeout=60) as response:
    result = json.load(response)
result

{'executed_at': '2026-08-27T06:34:50.568Z',
 'query': 'checkout android gen-z',
 'method': 'lakebase_text BM25',
 'search_function': 'app.search_experiments',
 'source_table': 'nimbus_serving.experiments',
 'source_kind': 'Build 1 continuous sync',
 'read_only': True,
 'index': 'experiments_description_bm25_idx',
 'execution_plan': ['Limit  (cost=5.38..5.39 rows=2 width=199)',
  '  ->  Sort  (cost=5.38..5.39 rows=2 width=199)',
  '        Sort Key: search_experiments.relevance DESC NULLS LAST',
  '        ->  Subquery Scan on search_experiments  (cost=4.27..5.37 rows=2 width=199)',
  '              ->  Limit  (cost=4.27..5.35 rows=2 width=207)',
  '                    ->  Result  (cost=4.27..5.35 rows=2 width=207)',
  '                          ->  Incremental Sort  (cost=4.27..4.32 rows=2 width=199)',
  '                                Sort Key: ((to_tsvector(\'english\'::regconfig, COALESCE(e.description, \'\'::text)) <@> \'("\'\'android\'\':2 \'\'checkout\'\':1 \'\'gen\'\':4 \'\'gen

In [2]:
plan = "\n".join(result["execution_plan"])
ids = [row["experiment_id"] for row in result["rows"]]
assert result["source_table"] == "nimbus_serving.experiments"
assert result["search_function"] == "app.search_experiments"
assert result["index"] == "experiments_description_bm25_idx"
assert result["read_only"] is True
assert "Index Scan" in plan
assert "EXP-0000009" in ids
print(f"executed_at: {result['executed_at']}")
print(f"path: {result['source_table']} -> {result['search_function']} -> {result['index']} -> Index Scan -> EXP-0000009")
print(f"rows: {ids}")
print("SEARCH_EXECUTION_VERIFIED")

executed_at: 2026-08-27T06:34:50.568Z
path: nimbus_serving.experiments -> app.search_experiments -> experiments_description_bm25_idx -> Index Scan -> EXP-0000009
rows: ['EXP-0000009', 'EXP-0000020', 'EXP-0000045', 'EXP-0000060', 'EXP-0000047']
SEARCH_EXECUTION_VERIFIED
